# Exploratory Data Analysis — Churn Prediction
## HPB Fintech Hackathon 2026

### Approach
We predict customer churn using a **temporal holdout** design:
- **9 months** of behavioral data → features
- **3 months** of churn observation → labels
- **2 non-overlapping periods** to maximize training data

| Period | Feature Window | Reference Date | Churn Window |
|--------|---------------|----------------|--------------|
| **P1** | Jul 2024 – Mar 2025 | 1 Apr 2025 | Apr – Jun 2025 |
| **P2** | Apr 2025 – Dec 2025 | 1 Jan 2026 | Jan – Mar 2026 |

**Churn** = client had active products at the reference date but has **zero** active products at the end of the churn window.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

COLORS = {'active': '#2ecc71', 'churned': '#e74c3c', 'primary': '#3498db', 'accent': '#f39c12'}
DATA = Path('../data')

# ── EU-format parsing helpers ──
def parse_eu_date(s):
    return pd.to_datetime(s.str.strip().str.rstrip('/'), format='%d/%m/%Y', errors='coerce')

def parse_eu_datetime(s):
    return pd.to_datetime(
        s.str.strip().str.replace('/ ', ' ', regex=False),
        format='%d/%m/%Y %H:%M:%S', errors='coerce')

def eu_to_float(s):
    return pd.to_numeric(
        s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce')

print('Libraries loaded')

In [ ]:
# ── Load all raw datasets ──
klijenti = pd.read_csv(DATA / 'klijenti.csv', dtype=str)
klijenti['DOB'] = pd.to_numeric(klijenti['DOB'], errors='coerce')
klijenti['DATUM_PRVOG_POCETKA'] = parse_eu_date(klijenti['DATUM_PRVOG_POCETKA_POSLOVNOG_ODNOSA'])

proizvodi = pd.read_csv(DATA / 'proizvodi.csv', dtype=str)
proizvodi['DATUM_OTVARANJA'] = parse_eu_date(proizvodi['DATUM_OTVARANJA'])
proizvodi['DATUM_ZATVARANJA'] = parse_eu_date(proizvodi['DATUM_ZATVARANJA'])

transakcije = pd.read_csv(DATA / 'transakcije.csv', dtype=str)
transakcije = transakcije.loc[:, transakcije.columns.notna()]
transakcije['DATE'] = parse_eu_datetime(transakcije['DATUM_I_VRIJEME_TRANSAKCIJE'])
transakcije['AMOUNT'] = eu_to_float(transakcije['IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI'])

stanja = pd.read_csv(DATA / 'stanja.csv', dtype=str)
stanja['STANJE'] = eu_to_float(stanja['STANJE_U_DOMICILNOJ_VALUTI'])
stanja['VRIJEDI_OD'] = parse_eu_date(stanja['VRIJEDI_OD'])

kontakt = pd.read_csv(DATA / 'kontakt.csv', dtype=str)
kontakt['DATE'] = parse_eu_datetime(kontakt['VRIJEME_KREIRANJA'])

# Map products → clients (for transaction/balance lookups later)
prod_client = proizvodi[['IDENTIFIKATOR_PROIZVODA', 'IDENTIFIKATOR_KLIJENTA']].drop_duplicates()

for name, df in [('Clients', klijenti), ('Products', proizvodi),
                  ('Transactions', transakcije), ('Balances', stanja),
                  ('Contact Center', kontakt)]:
    print(f'{name:20s} {df.shape[0]:>10,} rows x {df.shape[1]:>3} cols')

## 1. Data Coverage & Temporal Constraints
Understanding which data sources limit our temporal windows.

In [ ]:
ranges = {
    'Client relationships': (klijenti['DATUM_PRVOG_POCETKA'].min(), klijenti['DATUM_PRVOG_POCETKA'].max()),
    'Product openings':     (proizvodi['DATUM_OTVARANJA'].min(), proizvodi['DATUM_OTVARANJA'].max()),
    'Transactions':         (transakcije['DATE'].min(), transakcije['DATE'].max()),
    'Balances':             (stanja['VRIJEDI_OD'].min(), stanja['VRIJEDI_OD'].max()),
    'Contact Center':       (kontakt['DATE'].min(), kontakt['DATE'].max()),
}
print('Date ranges per data source:')
for name, (mn, mx) in ranges.items():
    print(f'  {name:25s}  {mn.date()}  ->  {mx.date()}')

fig, ax = plt.subplots(figsize=(14, 3.5))
colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
for i, (name, (mn, mx)) in enumerate(ranges.items()):
    ax.barh(i, mdates.date2num(mx) - mdates.date2num(mn),
            left=mdates.date2num(mn), color=colors_bar[i], alpha=0.85, height=0.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.set_yticks(range(len(ranges)))
ax.set_yticklabels(list(ranges.keys()), fontsize=11)
ax.axvline(mdates.date2num(pd.Timestamp('2024-07-01')), color='red', ls='--', lw=1.5,
           label='Full data available from Jul 2024')
ax.legend(loc='lower right', fontsize=10)
ax.set_title('Data Availability Timeline', fontweight='bold', fontsize=13)
sns.despine(left=True)
plt.tight_layout()
plt.show()

print()
print('Transactions start June 2024 - this is our binding constraint.')
print('All behavioral features must use data from July 2024 onward.')

## 2. Churn Definition & Population

We use **Period 2** (reference date: Jan 1, 2026) for all EDA visualizations, since it represents the most recent client state.

**Churn definition:**
- **At-risk**: client had at least 1 active product on the reference date
- **Churned**: client has **zero** active products at the end of the 3-month churn window
- Products with future expiry/closure dates are treated as **active** (not closed)

In [ ]:
# ── Compute churn labels for EDA (Period 2) ──
REF_VIZ = pd.Timestamp('2026-01-01')
CHURN_END_VIZ = pd.Timestamp('2026-03-31')

# Products active at reference date
prods_at_ref = proizvodi[
    (proizvodi['DATUM_OTVARANJA'] <= REF_VIZ) &
    (proizvodi['DATUM_ZATVARANJA'].isna() | (proizvodi['DATUM_ZATVARANJA'] > REF_VIZ))
]
at_risk_ids = set(prods_at_ref['IDENTIFIKATOR_KLIJENTA'].unique())

# Products active at churn end (including any opened after ref)
prods_at_end = proizvodi[
    (proizvodi['DATUM_OTVARANJA'] <= CHURN_END_VIZ) &
    (proizvodi['DATUM_ZATVARANJA'].isna() | (proizvodi['DATUM_ZATVARANJA'] > CHURN_END_VIZ))
]
active_at_end_ids = set(prods_at_end['IDENTIFIKATOR_KLIJENTA'].unique())

# Churned = at-risk but no products left at churn end
churned_ids = at_risk_ids - active_at_end_ids
active_ids = at_risk_ids & active_at_end_ids

labels = pd.DataFrame({
    'IDENTIFIKATOR_KLIJENTA': list(at_risk_ids),
    'churned': [1 if c in churned_ids else 0 for c in at_risk_ids]
})

# Merge with client data
eda_df = klijenti.merge(labels, on='IDENTIFIKATOR_KLIJENTA')
eda_df['tenure_years'] = (REF_VIZ - eda_df['DATUM_PRVOG_POCETKA']).dt.days / 365.25

n_c = labels['churned'].sum()
n_a = len(labels) - n_c
print(f'At-risk population (P2): {len(labels):,}')
print(f'  Churned: {n_c:,} ({n_c/len(labels):.1%})')
print(f'  Active:  {n_a:,} ({n_a/len(labels):.1%})')

## 3. Client Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Age distribution
for val, label, color in [(0, 'Active', COLORS['active']), (1, 'Churned', COLORS['churned'])]:
    subset = eda_df.loc[eda_df['churned'] == val, 'DOB'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.55, color=color, label=label, density=True)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Density')
axes[0].set_title('Age Distribution', fontweight='bold')
axes[0].legend()

# Tenure distribution
for val, label, color in [(0, 'Active', COLORS['active']), (1, 'Churned', COLORS['churned'])]:
    subset = eda_df.loc[eda_df['churned'] == val, 'tenure_years'].dropna()
    axes[1].hist(subset, bins=30, alpha=0.55, color=color, label=label, density=True)
axes[1].set_xlabel('Tenure (years)')
axes[1].set_title('Client Tenure', fontweight='bold')
axes[1].legend()

# Salary at bank - churn rate
salary = eda_df.groupby('KLIJENT_PRIMA_OSNOVNO_PRIMANJE_U_BANCI')['churned'].agg(['mean', 'size']).reset_index()
salary.columns = ['Salary at Bank', 'Churn Rate', 'Count']
bars = axes[2].bar(salary['Salary at Bank'].fillna('N/A'), salary['Churn Rate'],
                   color=[COLORS['churned'], COLORS['active']], alpha=0.8)
axes[2].set_ylabel('Churn Rate')
axes[2].set_title('Churn Rate by Salary Status', fontweight='bold')
for bar, rate in zip(bars, salary['Churn Rate']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{rate:.1%}', ha='center', fontweight='bold')

plt.suptitle('Client Demographics vs Churn', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Product Portfolio

In [ ]:
# Products per client at reference date
n_prods = prods_at_ref.groupby('IDENTIFIKATOR_KLIJENTA').size().reset_index(name='n_products')
prod_df = labels.merge(n_prods, on='IDENTIFIKATOR_KLIJENTA', how='left')
prod_df['n_products'] = prod_df['n_products'].fillna(0).astype(int)

# Has credit product
kredit_clients = set(
    prods_at_ref[prods_at_ref['NAZIV_DOMENE_PROIZVODA'] == 'KREDITI']['IDENTIFIKATOR_KLIJENTA'])
prod_df['has_kredit'] = prod_df['IDENTIFIKATOR_KLIJENTA'].isin(kredit_clients).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Product count by churn
for val, label, color in [(0, 'Active', COLORS['active']), (1, 'Churned', COLORS['churned'])]:
    subset = prod_df[prod_df['churned'] == val]['n_products']
    axes[0].hist(subset, bins=range(0, subset.max()+2), alpha=0.55, color=color, label=label, density=True)
axes[0].set_xlabel('Number of Active Products')
axes[0].set_ylabel('Density')
axes[0].set_title('Product Count Distribution', fontweight='bold')
axes[0].legend()

# Credit product vs churn
kredit_churn = prod_df.groupby('has_kredit')['churned'].agg(['mean', 'size']).reset_index()
kredit_churn.columns = ['Has Credit', 'Churn Rate', 'Count']
bars = axes[1].bar(['No Credit', 'Has Credit'], kredit_churn['Churn Rate'],
                   color=[COLORS['accent'], COLORS['primary']], alpha=0.8)
axes[1].set_ylabel('Churn Rate')
axes[1].set_title('Churn Rate by Credit Product', fontweight='bold')
for bar, rate in zip(bars, kredit_churn['Churn Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{rate:.1%}', ha='center', fontweight='bold')

plt.suptitle('Product Portfolio vs Churn', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Transaction Behavior (Apr – Dec 2025)

In [ ]:
# Transactions in P2 feature window
txn_w = transakcije[(transakcije['DATE'] >= '2025-04-01') & (transakcije['DATE'] <= '2025-12-31')]
txn_w = txn_w.merge(prod_client, on='IDENTIFIKATOR_PROIZVODA', how='left')
txn_w = txn_w[txn_w['IDENTIFIKATOR_KLIJENTA'].isin(at_risk_ids)]
txn_w = txn_w.merge(labels, on='IDENTIFIKATOR_KLIJENTA', how='left')

# Monthly transaction counts by churn status
txn_w['month'] = txn_w['DATE'].dt.to_period('M')
monthly = txn_w.groupby(['month', 'churned']).size().reset_index(name='n_txn')

# Normalize by client count per group
n_per_group = labels['churned'].value_counts().to_dict()
monthly['avg_per_client'] = monthly.apply(
    lambda r: r['n_txn'] / n_per_group.get(r['churned'], 1), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for val, label, color in [(0, 'Active', COLORS['active']), (1, 'Churned', COLORS['churned'])]:
    sub = monthly[monthly['churned'] == val].sort_values('month')
    axes[0].plot(sub['month'].astype(str), sub['avg_per_client'],
                 marker='o', color=color, label=label, linewidth=2)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Avg Transactions per Client')
axes[0].set_title('Monthly Transaction Activity', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Transaction amount distribution
for val, label, color in [(0, 'Active', COLORS['active']), (1, 'Churned', COLORS['churned'])]:
    subset = txn_w.loc[txn_w['churned'] == val, 'AMOUNT'].dropna()
    subset = subset[(subset > 0) & (subset < subset.quantile(0.99))]
    axes[1].hist(subset, bins=50, alpha=0.5, color=color, label=label, density=True)
axes[1].set_xlabel('Transaction Amount (EUR)')
axes[1].set_ylabel('Density')
axes[1].set_title('Transaction Amount Distribution', fontweight='bold')
axes[1].legend()

plt.suptitle('Transaction Behavior vs Churn', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Summary stats
txn_per_client = txn_w.groupby(['IDENTIFIKATOR_KLIJENTA', 'churned']).agg(
    n_txn=('AMOUNT', 'size'), avg_amt=('AMOUNT', 'mean')).reset_index()
print('Transaction summary (P2 feature window):')
for val, label in [(0, 'Active'), (1, 'Churned')]:
    sub = txn_per_client[txn_per_client['churned'] == val]
    print(f'  {label}: median {sub["n_txn"].median():.0f} txns, '
          f'avg amount {sub["avg_amt"].median():.0f} EUR')

## 6. Feature Engineering

We compute **12 features** for each temporal period, using only data available **before** the reference date.

| # | Feature | Source | Description |
|---|---------|--------|-------------|
| 1 | `dob` | Clients | Age of client |
| 2 | `tenure_years` | Clients | Years since first relationship (at ref date) |
| 3 | `n_products` | Products | Active products at ref date |
| 4 | `has_kredit` | Products | Has active credit product at ref date |
| 5 | `prima_placu` | Clients | Receives salary at bank (snapshot*) |
| 6 | `avg_txn_per_month` | Transactions | Avg monthly transaction count in feature window |
| 7 | `avg_txn_amount` | Transactions | Mean transaction amount in feature window |
| 8 | `txn_trend` | Transactions | Slope of monthly transaction counts |
| 9 | `avg_balance` | Balances | Mean account balance in feature window |
| 10 | `balance_trend` | Balances | Slope of monthly median balances |
| 11 | `has_contact` | Contact Center | Any interaction in feature window |
| 12 | `n_contacts` | Contact Center | Count of interactions in feature window |

*`prima_placu` is a current snapshot field — slight leakage risk for churned clients noted.

In [ ]:
def compute_period(name, feature_start, feature_end, ref_date, churn_end):
    """Compute features and labels for one temporal period."""
    fs = pd.Timestamp(feature_start)
    fe = pd.Timestamp(feature_end)
    ref = pd.Timestamp(ref_date)
    ce = pd.Timestamp(churn_end)

    # ── Population & Labels ──
    prods_ref = proizvodi[
        (proizvodi['DATUM_OTVARANJA'] <= ref) &
        (proizvodi['DATUM_ZATVARANJA'].isna() | (proizvodi['DATUM_ZATVARANJA'] > ref))
    ]
    risk_ids = set(prods_ref['IDENTIFIKATOR_KLIJENTA'].unique())

    prods_end = proizvodi[
        (proizvodi['DATUM_OTVARANJA'] <= ce) &
        (proizvodi['DATUM_ZATVARANJA'].isna() | (proizvodi['DATUM_ZATVARANJA'] > ce))
    ]
    active_end_ids = set(prods_end['IDENTIFIKATOR_KLIJENTA'].unique())
    churned = risk_ids - active_end_ids

    feat = pd.DataFrame({'IDENTIFIKATOR_KLIJENTA': list(risk_ids)})
    feat['churned'] = feat['IDENTIFIKATOR_KLIJENTA'].isin(churned).astype(int)
    feat['period'] = name

    # ── Static Features ──
    feat = feat.merge(klijenti[['IDENTIFIKATOR_KLIJENTA', 'DOB']], on='IDENTIFIKATOR_KLIJENTA', how='left')
    kl = klijenti[['IDENTIFIKATOR_KLIJENTA', 'DATUM_PRVOG_POCETKA']].copy()
    kl['tenure_years'] = (ref - kl['DATUM_PRVOG_POCETKA']).dt.days / 365.25
    feat = feat.merge(kl[['IDENTIFIKATOR_KLIJENTA', 'tenure_years']], on='IDENTIFIKATOR_KLIJENTA', how='left')

    n_pr = prods_ref.groupby('IDENTIFIKATOR_KLIJENTA').size().reset_index(name='n_products')
    feat = feat.merge(n_pr, on='IDENTIFIKATOR_KLIJENTA', how='left')

    kredit_ids = set(prods_ref[prods_ref['NAZIV_DOMENE_PROIZVODA'] == 'KREDITI']['IDENTIFIKATOR_KLIJENTA'])
    feat['has_kredit'] = feat['IDENTIFIKATOR_KLIJENTA'].isin(kredit_ids).astype(int)

    feat = feat.merge(
        klijenti[['IDENTIFIKATOR_KLIJENTA', 'KLIJENT_PRIMA_OSNOVNO_PRIMANJE_U_BANCI']],
        on='IDENTIFIKATOR_KLIJENTA', how='left')
    feat['prima_placu'] = (feat['KLIJENT_PRIMA_OSNOVNO_PRIMANJE_U_BANCI'] == 'DA').astype(int)
    feat.drop(columns=['KLIJENT_PRIMA_OSNOVNO_PRIMANJE_U_BANCI'], inplace=True)

    # ── Transaction Features ──
    txn_w = transakcije[(transakcije['DATE'] >= fs) & (transakcije['DATE'] <= fe)]
    txn_c = txn_w.merge(prod_client, on='IDENTIFIKATOR_PROIZVODA', how='left')
    txn_c = txn_c[txn_c['IDENTIFIKATOR_KLIJENTA'].isin(risk_ids)]

    txn_agg = txn_c.groupby('IDENTIFIKATOR_KLIJENTA').agg(
        n_txn=('AMOUNT', 'size'),
        avg_txn_amount=('AMOUNT', 'mean'),
        n_months=('DATE', lambda x: x.dt.to_period('M').nunique())
    ).reset_index()
    txn_agg['avg_txn_per_month'] = txn_agg['n_txn'] / txn_agg['n_months'].clip(lower=1)
    feat = feat.merge(txn_agg[['IDENTIFIKATOR_KLIJENTA', 'avg_txn_per_month', 'avg_txn_amount']],
                      on='IDENTIFIKATOR_KLIJENTA', how='left')

    # Transaction trend (slope of monthly counts)
    txn_mc = txn_c.groupby(
        ['IDENTIFIKATOR_KLIJENTA', txn_c['DATE'].dt.to_period('M')]
    ).size().reset_index(name='n_txn')
    txn_mc['mo'] = txn_mc['DATE'].apply(lambda p: p.year * 12 + p.month)

    def slope(grp):
        if len(grp) < 2:
            return 0.0
        x = grp['mo'].values.astype(float)
        y = grp['n_txn'].values.astype(float)
        x = x - x.mean()
        d = (x ** 2).sum()
        return (x * (y - y.mean())).sum() / d if d else 0.0

    txn_slopes = txn_mc.groupby('IDENTIFIKATOR_KLIJENTA').apply(slope).reset_index(name='txn_trend')
    feat = feat.merge(txn_slopes, on='IDENTIFIKATOR_KLIJENTA', how='left')

    # ── Balance Features ──
    bal_w = stanja[
        (stanja['TIP_STANJA'] == 'STANJE_RACUNA') &
        (stanja['VRIJEDI_OD'] >= fs) & (stanja['VRIJEDI_OD'] <= fe)
    ]
    bal_c = bal_w.merge(prod_client, on='IDENTIFIKATOR_PROIZVODA', how='left')
    bal_c = bal_c[bal_c['IDENTIFIKATOR_KLIJENTA'].isin(risk_ids)]

    avg_b = bal_c.groupby('IDENTIFIKATOR_KLIJENTA')['STANJE'].mean().reset_index(name='avg_balance')
    feat = feat.merge(avg_b, on='IDENTIFIKATOR_KLIJENTA', how='left')

    bal_c = bal_c.copy()
    bal_c['mo'] = bal_c['VRIJEDI_OD'].dt.year * 12 + bal_c['VRIJEDI_OD'].dt.month
    bal_mo = bal_c.groupby(['IDENTIFIKATOR_KLIJENTA', 'mo'])['STANJE'].median().reset_index(name='n_txn')

    bal_slopes = bal_mo.groupby('IDENTIFIKATOR_KLIJENTA').apply(slope).reset_index(name='balance_trend')
    feat = feat.merge(bal_slopes, on='IDENTIFIKATOR_KLIJENTA', how='left')

    # ── Contact Features ──
    kon_w = kontakt[(kontakt['DATE'] >= fs) & (kontakt['DATE'] <= fe)]
    kon_w = kon_w[kon_w['IDENTIFIKATOR_KLIJENTA'].isin(risk_ids)]
    contact_ids = set(kon_w['IDENTIFIKATOR_KLIJENTA'].dropna())
    feat['has_contact'] = feat['IDENTIFIKATOR_KLIJENTA'].isin(contact_ids).astype(int)
    n_kon = kon_w.groupby('IDENTIFIKATOR_KLIJENTA').size().reset_index(name='n_contacts')
    feat = feat.merge(n_kon, on='IDENTIFIKATOR_KLIJENTA', how='left')

    return feat


PERIODS = [
    ('P1', '2024-07-01', '2025-03-31', '2025-04-01', '2025-06-30'),
    ('P2', '2025-04-01', '2025-12-31', '2026-01-01', '2026-03-31'),
]

all_dfs = []
for name, fs, fe, ref, ce in PERIODS:
    df_p = compute_period(name, fs, fe, ref, ce)
    all_dfs.append(df_p)
    n_c = df_p['churned'].sum()
    print(f'{name}: {len(df_p):,} at-risk clients, {n_c:,} churned ({df_p["churned"].mean():.1%})')

combined = pd.concat(all_dfs, ignore_index=True)
print(f'\nCombined: {len(combined):,} rows, churn rate {combined["churned"].mean():.1%}')

## 7. Feature Summary & Export

In [ ]:
FEATURE_COLS = ['dob', 'tenure_years', 'n_products', 'has_kredit', 'prima_placu',
                'avg_txn_per_month', 'avg_txn_amount', 'txn_trend',
                'avg_balance', 'balance_trend', 'has_contact', 'n_contacts']

# Rename DOB column for consistency
combined.rename(columns={'DOB': 'dob'}, inplace=True)

print('Feature statistics:')
print(combined[FEATURE_COLS + ['churned']].describe().round(3).to_string())
print(f'\nMissing values:')
print(combined[FEATURE_COLS].isnull().sum().to_string())

# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = combined[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, vmin=-1, vmax=1, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Max mutual correlation
for i in range(len(corr)):
    corr.iloc[i, i] = 0.0
mc = corr.abs().max().max()
pair = corr.abs().stack().idxmax()
print(f'\nMax mutual |correlation|: {mc:.3f} ({pair[0]} <-> {pair[1]})')

# Export
out = combined[['IDENTIFIKATOR_KLIJENTA', 'period'] + FEATURE_COLS + ['churned']]
out_path = DATA / 'churn_features_raw.csv'
out.to_csv(out_path, index=False)
print(f'\nExported: {out_path}')
print(f'  Shape: {out.shape}')